In [20]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

In [21]:
class TrafficSignDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        img_path = self.annotations.iloc[idx]['Path']
        full_path = os.path.abspath(img_path)
        
        image = Image.open(full_path).convert('RGB')
        
        y_label = torch.tensor(int(self.annotations.iloc[idx]['ClassId']))
        
        if self.transform:
            image = self.transform(image)
            
        return (image, y_label)

In [22]:
# Testing DataLoader and TestLoader
if __name__ == '__main__':
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor()
    ])
    
    print("Loading datasets...")
    
    train_dataset = TrafficSignDataset(csv_file='Train.csv', transform=transform)
    test_dataset = TrafficSignDataset(csv_file='Test.csv', transform=transform)
    
    batch_size = 64
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)
    
    print("Testing the DataLoader...")
    data_iter = iter(train_loader)
    images, labels = next(data_iter)
    
    print(f"Batch image shape: {images.shape} -> (Batch Size, Channels, Height, Width)")
    print(f"Batch label shape: {labels.shape}")

Loading datasets...
Testing the DataLoader...
Batch image shape: torch.Size([64, 3, 32, 32]) -> (Batch Size, Channels, Height, Width)
Batch label shape: torch.Size([64])


In [23]:
import torch.nn as nn
import torch.nn.functional as F

class TrafficSignNet(nn.Module):
    def __init__(self, num_classes=43):
        super(TrafficSignNet, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = torch.flatten(x, 1)
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

In [25]:
print("\nTesting the DataLoader...")
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Batch image shape: {images.shape}")
print(f"Batch label shape: {labels.shape}")

print(f"\nTesting the Neural Network...")

model = TrafficSignNet(num_classes=43)

predictions = model(images)

print(f"Network Output Shape: {predictions.shape} -> (Batch Size, Number of Classes)")


Testing the DataLoader...
Batch image shape: torch.Size([64, 3, 32, 32])
Batch label shape: torch.Size([64])

Testing the Neural Network...
Network Output Shape: torch.Size([64, 43]) -> (Batch Size, Number of Classes)


In [26]:
import torch.optim as optim

if __name__ == "__main__":
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor() 
    ])
    
    print("Loading datasets...")
    train_dataset = TrafficSignDataset(csv_file='Train.csv', transform=transform)
    train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)

    print("Initializing model...")
    model = TrafficSignNet(num_classes=43).to(device)
    
    criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 10
    print("\nStarting Training... (This may take a few minutes)")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            
            # --- THE CORE LEARNING PROCESS ---
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            
            optimizer.step()
            
            # --- TRACKING PROGRESS ---
            running_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

    print("\nTraining Complete!")
    
    # Save the trained model to your hard drive!
    torch.save(model.state_dict(), 'traffic_sign_model.pth')
    print("Model saved to 'traffic_sign_model.pth'")

Using device: mps
Loading datasets...
Initializing model...

Starting Training... (This may take a few minutes)
Epoch [1/10] | Loss: 2.2379 | Accuracy: 34.71%
Epoch [2/10] | Loss: 0.6418 | Accuracy: 78.56%
Epoch [3/10] | Loss: 0.2913 | Accuracy: 90.75%
Epoch [4/10] | Loss: 0.1776 | Accuracy: 94.39%
Epoch [5/10] | Loss: 0.1298 | Accuracy: 95.97%
Epoch [6/10] | Loss: 0.0989 | Accuracy: 96.93%
Epoch [7/10] | Loss: 0.0796 | Accuracy: 97.39%
Epoch [8/10] | Loss: 0.0646 | Accuracy: 97.97%
Epoch [9/10] | Loss: 0.0512 | Accuracy: 98.38%
Epoch [10/10] | Loss: 0.0509 | Accuracy: 98.33%

Training Complete!
Model saved to 'traffic_sign_model.pth'
